# Seleção das Imagens com Linhas dos Datasets
***

### Importação dos Pacotes Utilitários
***

In [1]:
from collections import Counter
import json, os
import utils

### Seleção das Imagens com Linhas de Transmissão: ```TTPLA Dataset```
***

In [6]:
# carregando os arquivos .json em uma lista
labels_folder = '../datasets-descompactados/data_original_size/'
jsons_files = utils.list_json_files(labels_folder)
jsons_files = sorted(jsons_files, key = utils.extract_number)

In [7]:
# contando a quantidade de objetos instanciados por classe em cada uma das imagens
count_category_counts = list()

for json_file in jsons_files:
    category_counts = utils.count_categories('../datasets-descompactados/data_original_size/' + json_file)
    count_category_counts.append(category_counts)

In [8]:
# cria uma lista com o nome das imagens que possuem linhas de transmissão
images_with_towers = list()

for category in count_category_counts:
    keys = category.keys()
    for key in keys:
        count = category[key]
        if key == 'cable': 
            images_with_towers.append(category['imagePath'])
        break
        
print(f'{len(images_with_towers)} Imagens que contém linhas de transmissão')

846 Imagens que contém linhas de transmissão


In [9]:
# copiando para uma pasta em separado as imagens que apenas contém linhas
source_path = '../datasets-descompactados/data_original_size/'
destination_path = 'TTPLA-dataset/'
utils.copy_archives(source_path, images_with_towers, destination_path)

Arquivos copiados com sucesso!


In [10]:
# pegando a lista de jsons das imagens selecionadas
jsons_with_towers = [images.replace(".jpg", ".json") for images in images_with_towers]

In [11]:
# copiando para uma pasta em separado os arquivos jsons das imagens que contém linhas
utils.copy_archives(source_path, jsons_with_towers, destination_path)

Arquivos copiados com sucesso!


In [12]:
# listando os arquivos jsons das imagens selecionadas
dir_files = 'TTPLA-dataset'
extension = '.json'
list_jsons = [archive for archive in os.listdir(dir_files) if archive.endswith(extension)]

print(f'FIles .json: {len(list_jsons)} files selected.')

FIles .json: 846 files selected.


In [13]:
# tratando o arquivo json para remoção de atributos não úteis
for json_file in list_jsons:
    content_json = utils.load_json_file('TTPLA-dataset/' + json_file)
    content_json['imageData'] = 'null'

    with open('TTPLA-dataset/' + json_file, 'w') as archive:
        json.dump(content_json, archive, indent = 4)

In [14]:
# selecionando nos arquivos json apenas as classes de linhas
new_shapes = list()
for json_file in list_jsons:
    content_json = utils.load_json_file('TTPLA-dataset/' + json_file)
    for shape in content_json['shapes']:
        if shape['label'] == 'cable':
            new_shapes.append(shape)
    
    content_json['shapes'] = new_shapes
    new_shapes = list()

    with open('TTPLA-dataset/' + json_file, 'w') as archive:
        json.dump(content_json, archive, indent = 4)

In [15]:
# normalizando o nome das classes de linhas de transmissão
for json_file in list_jsons:
    content_json = utils.load_json_file('TTPLA-dataset/' + json_file)
    for shape in content_json['shapes']:
        shape['label'] = 'lines'
    
    with open('TTPLA-dataset/' + json_file, 'w') as archive:
        json.dump(content_json, archive, indent = 4)